## UEFS – Curso de Programação em Python: A Invasão do Império – O Despertar do Guardião 
Por: Luan Almeida Meira com professor: Malachor Jonatas

In [1]:
# Biblioteca usadas para criar o jogo

import pygame
import sys
import random


# Jogo: O Despertar do Guardião feito por Luan Almeida Meira 
pygame.init()
pygame.mixer.init()

# Configurações da tela

LARGURA = 900
ALTURA = 700

tela = pygame.display.set_mode((LARGURA, ALTURA))
pygame.display.set_caption("O Despertar do Guardião")

clock = pygame.time.Clock()

# Cores

PRETO = (0, 0, 0)
BRANCO = (255, 255, 255)
AZUL = (50, 80, 255)
CINZA = (90, 90, 90)
VERDE = (0, 180, 0)
VERMELHO = (180, 0, 0)
AMARELO = (220, 220, 0)


# fontes

fonte = pygame.font.SysFont("arial", 30, True)
fonte_pequena = pygame.font.SysFont("arial", 22)
fonte_titulo = pygame.font.SysFont("arial", 55, True)

# Imagens dos objetos do jogo

img_nave = pygame.image.load("pngwing.com.png")
img_nave = pygame.transform.scale(img_nave, (60, 60))

img_vader = pygame.image.load("black-red-darth-vader-head-sticker.png")
img_vader = pygame.transform.scale(img_vader, (100, 100))

img_bacta = pygame.image.load("bactas.png")
img_bacta = pygame.transform.scale(img_bacta, (20, 20))
    
img_stormtroopers = pygame.image.load("stormtrooper.png")    
img_stormtroopers = pygame.transform.scale(img_stormtroopers, (50, 50))

# Imagens do fundo do jogo 

img_fundo_menu = pygame.image.load("fundo menu_pausa.jpg")
img_fundo_menu = pygame.transform.scale(img_fundo_menu, (LARGURA, ALTURA))

img_fundo_jogo = pygame.image.load("fundo.jpg")
img_fundo_jogo = pygame.transform.scale(img_fundo_jogo, (LARGURA, ALTURA))

img_fundo_pause = pygame.image.load("fundo menu_pausa.jpg")
img_fundo_pause = pygame.transform.scale(img_fundo_pause, (LARGURA, ALTURA))

img_fundo_vitoria = pygame.image.load("fundo vitoria.jpg")
img_fundo_vitoria = pygame.transform.scale(img_fundo_vitoria, (LARGURA, ALTURA))

player_tamanho = 60

# Sons do jogo

som_tiro = pygame.mixer.Sound("tiro.wav")
som_explosao = pygame.mixer.Sound("morte.wav")
som_bacta = pygame.mixer.Sound("coleta_bacta.mp3")
som_vitoria = pygame.mixer.Sound("vitoria.wav")
som_gameover = pygame.mixer.Sound("game_over.mp3")
som_botoes = pygame.mixer.Sound("botoes.ogg")
som_impacto = pygame.mixer.Sound("impacto.wav")

# Estados atuais do jogo

estado = "menu"

# Dificuldades 
dificuldades = ["Fácil", "Médio", "Difícil"]
indice_dificuldade = 1
mostrar_opcoes = False

# Limite de spawn/ dificuldades

spawn_limites = [90, 60, 35]

# Variaveis do jogo 

player_x = 450
player_y = 350
player_vel = 8

tiros = []
stormtroopers = []
tiros_inimigos = []
bactas = []
score = 0
vida = 3
tempo_spawn = 0

vader = None
vader_hp = [5, 10, 20]
vader_ativo = False
tiros_vader = []
tempo_inicio = pygame.time.get_ticks()

ultimo_tiro = 0
velocidades = [1, 2, 3]


# botões do menu

botao_iniciar = pygame.Rect(300, 220, 300, 50)
botao_dificuldade = pygame.Rect(300, 290, 300, 50)
botao_sair = pygame.Rect(300, 520, 300, 50)

botao_facil = pygame.Rect(300, 345, 300, 45)
botao_medio = pygame.Rect(300, 390, 300, 45)
botao_dificil = pygame.Rect(300, 435, 300, 45)


# botões da pausa

botao_continuar = pygame.Rect(300, 250, 300, 50)
botao_voltar = pygame.Rect(300, 340, 300, 50)
botao_sair_pause = pygame.Rect(300, 430, 300, 50)

# botoes do game over e a de vitória

botao_reiniciar = pygame.Rect(300, 380, 300, 50)
botao_menu_fim = pygame.Rect(300, 450, 300, 50)

# criação da função para deixar o texto centralizado

def texto_centro(msg, fonte, cor, y):
    render = fonte.render(msg, True, cor)
    rect = render.get_rect(center=(LARGURA // 2, y))
    tela.blit(render, rect)

# criação da função para resetar o jogo

def resetar_jogo():
    global player_x, player_y, tiros, stormtroopers, tiros_inimigos
    global bactas, score, vida, tempo_spawn, vader, vader_hp
    global vader_ativo, tiros_vader, tempo_inicio, ultimo_tiro, estado

    player_x = 450
    player_y = 350
    tiros = []
    stormtroopers = []
    tiros_inimigos = []
    bactas = []
    score = 0
    vida = 3
    tempo_spawn = 0
    vader = None
    vader_hp = vader_hp[indice_dificuldade]
    vader_ativo = False
    tiros_vader = []
    tempo_inicio = pygame.time.get_ticks()
    ultimo_tiro = 0
    estado = "jogando"

# loop principal do jogo
while True:

    mx, my = pygame.mouse.get_pos()

    for event in pygame.event.get():

        # Fechamendo do jogo
        if event.type == pygame.QUIT:
            pygame.quit()
            sys.exit()

        
        # Teclas do teclado - Movimento E Tiro

        if event.type == pygame.KEYDOWN:

            if estado == "jogando":

                # PAUSAR
                if event.key == pygame.K_ESCAPE:
                    estado = "pausa"

                # ATIRAR
                if event.key == pygame.K_SPACE:
                    agora = pygame.time.get_ticks()
                    if agora - ultimo_tiro > 250:
                        tiros.append(pygame.Rect(player_x + player_tamanho // 2 - 3, player_y, 6, 15))
                        som_tiro.play()
                        ultimo_tiro = agora

    
        # clique do mouse
        if event.type == pygame.MOUSEBUTTONDOWN:

            if estado == "menu":

                if botao_iniciar.collidepoint(mx, my):
                    som_botoes.play()
                    resetar_jogo()

                elif botao_sair.collidepoint(mx, my):
                    som_botoes.play()
                    pygame.quit()
                    sys.exit()

                elif botao_dificuldade.collidepoint(mx, my):
                    som_botoes.play()
                    mostrar_opcoes = not mostrar_opcoes

                if mostrar_opcoes:
                    if botao_facil.collidepoint(mx, my):
                        som_botoes.play()
                        indice_dificuldade = 0
                        mostrar_opcoes = False
                    elif botao_medio.collidepoint(mx, my):
                        som_botoes.play()
                        indice_dificuldade = 1
                        mostrar_opcoes = False
                    elif botao_dificil.collidepoint(mx, my):
                        som_botoes.play()
                        indice_dificuldade = 2
                        mostrar_opcoes = False

            elif estado == "pausa":

                if botao_continuar.collidepoint(mx, my):
                    som_botoes.play()
                    estado = "jogando"
                elif botao_voltar.collidepoint(mx, my):
                    som_botoes.play()
                    estado = "menu"
                    mostrar_opcoes = False
                elif botao_sair_pause.collidepoint(mx, my):
                    som_botoes.play()
                    pygame.quit()
                    sys.exit()

            elif estado in ("gameover", "vitoria"):

                if botao_reiniciar.collidepoint(mx, my):
                    som_botoes.play()
                    resetar_jogo()
                elif botao_menu_fim.collidepoint(mx, my):
                    som_botoes.play()
                    estado = "menu"
                    mostrar_opcoes = False

    # fundo do jogo
    if estado == "menu":
        tela.blit(img_fundo_menu, (0, 0))
    elif estado == "jogando":           
        tela.blit(img_fundo_jogo, (0, 0))
    elif estado == "pausa":
        tela.blit(img_fundo_pause, (0, 0))
    elif estado in ("gameover", "vitoria"):
        tela.blit(img_fundo_menu, (0, 0))

    # Menu principal do jogo

    if estado == "menu":

        texto_centro("MENU PRINCIPAL", fonte_titulo, AZUL, 100)
        texto_centro("Desenvolvido por Luan Almeida Meira", fonte_pequena, AMARELO, 150)

        pygame.draw.rect(tela, VERDE, botao_iniciar)
        texto_centro("INICIAR", fonte, BRANCO, 245)

        pygame.draw.rect(tela, CINZA, botao_dificuldade)
        texto_centro("Dificuldade: " + dificuldades[indice_dificuldade], fonte, BRANCO, 315)

        if mostrar_opcoes:
            pygame.draw.rect(tela, CINZA, botao_facil)
            pygame.draw.rect(tela, CINZA, botao_medio)
            pygame.draw.rect(tela, CINZA, botao_dificil)
            texto_centro("Fácil",   fonte, BRANCO, 367)
            texto_centro("Médio",   fonte, BRANCO, 412)
            texto_centro("Difícil", fonte, BRANCO, 457)

        pygame.draw.rect(tela, VERMELHO, botao_sair)
        texto_centro("SAIR", fonte, BRANCO, 545)

    elif estado == "jogando":

        teclas = pygame.key.get_pressed()

        # MOVIMENTO WASD + SETAS
        if teclas[pygame.K_w] or teclas[pygame.K_UP]:
            player_y -= player_vel
        if teclas[pygame.K_s] or teclas[pygame.K_DOWN]:
            player_y += player_vel
        if teclas[pygame.K_a] or teclas[pygame.K_LEFT]:
            player_x -= player_vel
        if teclas[pygame.K_d] or teclas[pygame.K_RIGHT]:
            player_x += player_vel

        # LIMITES DA TELA
        player_x = max(0, min(player_x, LARGURA - player_tamanho))
        player_y = max(0, min(player_y, ALTURA - player_tamanho))

        player_rect = pygame.Rect(player_x, player_y, player_tamanho, player_tamanho)

        # spawn do Darth Vader
        if score >= 200 and not vader_ativo:
            vader = pygame.Rect(LARGURA // 2 - 50, 50, 100, 100)
            vader_ativo = True

        
        # spawn dos STORMTROOPERS 
        if not vader_ativo:
            tempo_spawn += 1

            if tempo_spawn >= spawn_limites[indice_dificuldade]:
                tempo_spawn = 0
                stormtroopers.append(
                    pygame.Rect(
                        random.randint(0, LARGURA - 40),
                        -40,
                        40,
                        40
                    )
                )

        # mover tiros do player/ stormtroopers

        for tiro in tiros[:]:
            tiro.y -= 10
            if tiro.bottom < 0:
                tiros.remove(tiro)

        for inimigo in stormtroopers[:]:
            inimigo.y += velocidades[indice_dificuldade]

            # Tiro aleatório do inimigo
            if random.randint(1, 120) == 1:
                tiros_inimigos.append(
                    pygame.Rect(inimigo.centerx, inimigo.bottom, 5, 15)
                )
                som_tiro.play()

            if inimigo.top > ALTURA:
                stormtroopers.remove(inimigo)
            elif inimigo.colliderect(player_rect):
                som_impacto.play()
                vida -= 1
                stormtroopers.remove(inimigo)

        # mover e tiros dos inimigos
        for tiro_i in tiros_inimigos[:]:
            tiro_i.y += 5
            if tiro_i.top > ALTURA:
                tiros_inimigos.remove(tiro_i)
            elif tiro_i.colliderect(player_rect):
                som_explosao.play()
                vida -= 1
                tiros_inimigos.remove(tiro_i)

        # mover dos bactas
        for bacta in bactas[:]:
            bacta.y += 3
            if bacta.top > ALTURA:
                bactas.remove(bacta)
            elif bacta.colliderect(player_rect):
                som_bacta.play()
                vida = min(vida + 1, 10)  # Limite máximo de vida
                bactas.remove(bacta)

        # efeitos de colisão dos tiros do player com os inimigos
        for tiro in tiros[:]:
            for inimigo in stormtroopers[:]:
                if tiro.colliderect(inimigo):
                    som_explosao.play()
                    if tiro in tiros:
                        tiros.remove(tiro)
                    if inimigo in stormtroopers:
                        stormtroopers.remove(inimigo)
                        score += 10

                        if random.randint(1, 5) == 1:
                            bactas.append(pygame.Rect(inimigo.x, inimigo.y, 20, 20))
                    break

        # Vader 

        if vader_ativo and vader:
            # Vader persegue o player horizontalmente
            if vader.centerx < player_x + player_tamanho // 2:
                vader.x += 6
            else:
                vader.x -= 6

            # Vader não sai da tela
            vader.x = max(0, min(vader.x, LARGURA - vader.width))

            # Tiros do Vader
            if random.randint(1, 40) == 1:
                tiros_vader.append(
                    pygame.Rect(vader.centerx, vader.bottom, 8, 20)
                )
                som_tiro.play()

            # Tiros do player acertam Vader
            for tiro in tiros[:]:
                if tiro.colliderect(vader):
                    som_explosao.play()
                    tiros.remove(tiro)
                    vader_hp -= 1
                    if vader_hp <= 0:
                        pygame.mixer.stop()
                        som_vitoria.play()
                        score += 200
                        estado = "vitoria"

        # mover tiros do Vader

        for tv in tiros_vader[:]:
            tv.y += 8
            if tv.top > ALTURA:
                tiros_vader.remove(tv)
            elif tv.colliderect(player_rect):
                som_impacto.play()
                vida -= 2
                tiros_vader.remove(tv)

        # Morte do player
        
        if vida <= 0:
            som_gameover.play()
            estado = "gameover"
        

        # Imagens dos objetos no jogo 

        player_rect = pygame.Rect(player_x, player_y, player_tamanho, player_tamanho)
        tela.blit(img_nave, (player_x, player_y))

        for tiro in tiros:
            pygame.draw.rect(tela, AMARELO, tiro)

        for inimigo in stormtroopers:
            tela.blit(img_stormtroopers, (inimigo.x, inimigo.y))

        for tiro_i in tiros_inimigos:
            pygame.draw.rect(tela, BRANCO, tiro_i)

        for bacta in bactas:
            tela.blit(img_bacta, (bacta.x, bacta.y))

        if vader_ativo and vader:
            tela.blit(img_vader, (vader.x, vader.y))
            txt_vader_label = fonte_pequena.render("VADER", True, BRANCO)
            tela.blit(txt_vader_label, (vader.x + 20, vader.y - 20))

        for tv in tiros_vader:
            pygame.draw.rect(tela, (255, 100, 100), tv)

        # HUD
        txt_score = fonte_pequena.render(f"Score: {score}", True, BRANCO)
        txt_vida  = fonte_pequena.render(f"Vida: {vida}",   True, BRANCO)

        tempo_decorrido = (pygame.time.get_ticks() - tempo_inicio) // 1000
        minutos  = tempo_decorrido // 60
        segundos = tempo_decorrido % 60
        txt_tempo = fonte_pequena.render(f"Tempo: {minutos:02}:{segundos:02}", True, BRANCO)

        tela.blit(txt_score, (15, 15))
        tela.blit(txt_vida,  (15, 45))
        tela.blit(txt_tempo, (15, 75))

        if vader_ativo:
            txt_vader = fonte_pequena.render(f"Vader HP: {vader_hp}", True, VERMELHO)
            tela.blit(txt_vader, (700, 15))

    # jogo pausado

    elif estado == "pausa":

        texto_centro("JOGO PAUSADO", fonte_titulo, AZUL, 120)

        pygame.draw.rect(tela, VERDE, botao_continuar)
        texto_centro("CONTINUAR", fonte, BRANCO, 275)

        pygame.draw.rect(tela, CINZA, botao_voltar)
        texto_centro("VOLTAR AO MENU", fonte, BRANCO, 365)

        pygame.draw.rect(tela, VERMELHO, botao_sair_pause)
        texto_centro("SAIR", fonte, BRANCO, 455)

    # Game Over do player
    elif estado == "gameover":
        tela.fill(VERMELHO)
        texto_centro("GAME OVER", fonte_titulo, BRANCO, 220)
        texto_centro(f"Score Final: {score}", fonte, BRANCO, 300)

        pygame.draw.rect(tela, VERDE, botao_reiniciar)
        texto_centro("JOGAR NOVAMENTE", fonte, BRANCO, 405)

        pygame.draw.rect(tela, CINZA, botao_menu_fim)
        texto_centro("MENU PRINCIPAL", fonte, BRANCO, 475)

    # Vitoria do player

    elif estado == "vitoria":
        tela.blit(img_fundo_vitoria, (0, 0))
        texto_centro("VOCÊ DERROTOU DARTH VADER!", fonte_titulo, VERMELHO, 200)
        texto_centro(f"Score Final: {score}", fonte, VERMELHO, 300)

        pygame.draw.rect(tela, VERDE, botao_reiniciar)
        texto_centro("JOGAR NOVAMENTE", fonte, BRANCO, 405)

        pygame.draw.rect(tela, CINZA, botao_menu_fim)
        texto_centro("MENU PRINCIPAL", fonte, BRANCO, 475)
        

    # Atualizção de tela

    pygame.display.update()
    clock.tick(60)

pygame 2.6.1 (SDL 2.28.4, Python 3.12.11)
Hello from the pygame community. https://www.pygame.org/contribute.html


SystemExit: 

c:\Users\Luan\anaconda3\envs\cwq\Lib\site-packages\IPython\core\interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
